In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report, accuracy_score, roc_auc_score
from sklearn.model_selection import learning_curve, LeaveOneOut, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest, f_classif, RFE, RFECV, SelectFromModel, SelectFpr, VarianceThreshold, mutual_info_classif
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier

In [2]:
# On peut mieux faire, car sur l'old metabolome on a un meileur score

In [17]:
dataset_total_path = "C:/Users/tamer/Documents/PhD/ML/Metabolome_PCA_paul.xlsx"
dataset_val_path = "C:/Users/tamer/Documents/PhD/ML/Metabolome_PCA_saqib.xlsx"

#dataset_total_path = "C:/Users/tamer/Documents/PhD/ML/old_metabolome_paul.xlsx"
#dataset_val_path = "C:/Users/tamer/Documents/PhD/ML/old_metabolome_saqib.xlsx"

df = pd.read_excel(dataset_total_path)
df_val = pd.read_excel(dataset_val_path)

print(df.shape)
print(df_val.shape)

(24, 376)
(12, 376)


In [18]:
Class_A = 'LP'
Class_B = 'SP'

df = df[(df["Class"] == Class_A) | (df["Class"] == Class_B)]
df_val = df_val[(df_val["Class"] == Class_A) | (df_val["Class"] == Class_B)]

In [19]:
def encodage(df):
    code = {
    'LP' : 1,
    'SP' : 0,
    'LN' : 3,
    'SN' : 4
}
# Appliquer ce dictionnaire aux colonnes du dataset, avec la fonction map
    for col in df.select_dtypes('object'):
        df[col] = df[col].map(code)

    return df


def preprocessing(df):
    df = encodage(df)

    X = df.drop(['Class'], axis = 1)
    y = df['Class']

    # compter le nombre d'échantillons restants dans le dataset après avoir été inputé
    print(y.value_counts())

    return X, y

In [20]:
X_total, y_total = preprocessing(df)
X_val, y_val = preprocessing(df_val)

Class
1    6
0    6
Name: count, dtype: int64
Class
0    3
1    3
Name: count, dtype: int64


In [21]:
print("var before log2 transormation : " , X_total.var(axis=0).mean())
log2_transformer = FunctionTransformer(lambda x: np.log2(x + 1))
X_total = log2_transformer.fit_transform(X_total)
X_val = log2_transformer.fit_transform(X_val)
print("var after log2 transormation : " , X_total.var(axis=0).mean())

var before log2 transormation :  7.684178425552448e+16
var after log2 transormation :  5.49605691519022


In [22]:
def evaluation(model):
    model.fit(X_total, y_total)
    ypred = model.predict(X_val)

    print(confusion_matrix(y_val, ypred))
    print(classification_report(y_val, ypred))

In [9]:
def evaluation_2(model):
    model.fit(X_total, y_total)
    ypred = model.predict(X_val)

    print(accuracy_score(y_val, ypred))

# Modèle 1

In [10]:
vars_ = X_total.var(axis=0)
q1 = np.quantile(vars_, 0.25) 
selector = VarianceThreshold(threshold = q1)
preprocessor = make_pipeline(selector)

#preprocessor = make_pipeline(SelectKBest(k=300))
#preprocessor = make_pipeline(SelectKBest(score_func=f_classif, k=155))
    

LR_L2 = make_pipeline(preprocessor, StandardScaler(), LogisticRegression(penalty='l2', random_state = 0, solver = 'liblinear'))
LR_L1 = make_pipeline(preprocessor, StandardScaler(), LogisticRegression(penalty='l1', random_state = 0, solver = 'liblinear'))
LR_EN = make_pipeline(preprocessor, StandardScaler(), LogisticRegression(penalty='elasticnet', l1_ratio = 0.5, random_state = 0, solver = 'saga'))
evaluation(LR_L1)
evaluation(LR_L2)
evaluation(LR_EN)

[[0 3]
 [0 3]]
              precision    recall  f1-score   support

           1       0.00      0.00      0.00         3
           3       0.50      1.00      0.67         3

    accuracy                           0.50         6
   macro avg       0.25      0.50      0.33         6
weighted avg       0.25      0.50      0.33         6

[[2 1]
 [0 3]]
              precision    recall  f1-score   support

           1       1.00      0.67      0.80         3
           3       0.75      1.00      0.86         3

    accuracy                           0.83         6
   macro avg       0.88      0.83      0.83         6
weighted avg       0.88      0.83      0.83         6

[[0 3]
 [0 3]]
              precision    recall  f1-score   support

           1       0.00      0.00      0.00         3
           3       0.50      1.00      0.67         3

    accuracy                           0.50         6
   macro avg       0.25      0.50      0.33         6
weighted avg       0.25      

C:\Users\tamer\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\tamer\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\tamer\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\tamer\anaconda3\Lib\site-packag

In [11]:
mask = LR_L2.named_steps['pipeline'].named_steps['variancethreshold'].get_support()
feat_names = X_total.columns[mask]
lr = LR_L2.named_steps['logisticregression']


coef = lr.coef_.ravel()  # shape: (n_selected_features,)
coef_lr_df = pd.DataFrame({'Feature': feat_names, 'Coefficient': coef}) \
            .sort_values('Coefficient', ascending=False)

coef_lr_df.to_excel("C:/Users/tamer/Documents/PhD/ML/ML_coefs/coefs_lr.xlsx")

# Modèle 2

In [12]:
vars_ = X_total.var(axis=0)
q1 = np.quantile(vars_, 0.75) 
selector = VarianceThreshold(threshold = q1)
#preprocessor = make_pipeline(selector)

#preprocessor = make_pipeline(SelectKBest(k=350))
preprocessor = SelectKBest(score_func=f_classif, k=105)


RandomForest = make_pipeline(preprocessor, RandomForestClassifier(random_state = 0))
AdaBoost = make_pipeline(preprocessor, AdaBoostClassifier(random_state = 0))
GB = make_pipeline(preprocessor, GradientBoostingClassifier(random_state = 0))

evaluation(RandomForest)
evaluation(AdaBoost)
evaluation(GB)

[[0 3]
 [0 3]]
              precision    recall  f1-score   support

           1       0.00      0.00      0.00         3
           3       0.50      1.00      0.67         3

    accuracy                           0.50         6
   macro avg       0.25      0.50      0.33         6
weighted avg       0.25      0.50      0.33         6

[[3 0]
 [0 3]]
              precision    recall  f1-score   support

           1       1.00      1.00      1.00         3
           3       1.00      1.00      1.00         3

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6

[[0 3]
 [0 3]]
              precision    recall  f1-score   support

           1       0.00      0.00      0.00         3
           3       0.50      1.00      0.67         3

    accuracy                           0.50         6
   macro avg       0.25      0.50      0.33         6
weighted avg       0.25      

C:\Users\tamer\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\tamer\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\tamer\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\tamer\anaconda3\Lib\site-packag

In [13]:
AdaBoost.fit(X_total, y_total)

selector = AdaBoost.named_steps["selectkbest"]
ada = AdaBoost.named_steps["adaboostclassifier"]

support_mask = selector.get_support()
selected_features = X_total.columns[support_mask]
importance_reduced = ada.feature_importances_

# ------------------------------------------------
# 5. Map importance back to full feature space
# ------------------------------------------------
importance_full = np.zeros(X_total.shape[1])
importance_full[support_mask] = importance_reduced

# ------------------------------------------------
# 6. Build results table
# ------------------------------------------------
results_df = pd.DataFrame({
    "metabolite": X_total.columns,
    "selected": support_mask,
    "importance": importance_full
}).sort_values("importance", ascending=False)

# ------------------------------------------------
# 7. Save to Excel
# ------------------------------------------------
results_df.to_excel("C:/Users/tamer/Documents/PhD/ML/ML_coefs/AdaBoost_Selected_Metabolites.xlsx", index=False)

print("Saved: AdaBoost_Selected_Metabolites.xlsx")

Saved: AdaBoost_Selected_Metabolites.xlsx


# Affinage de modèle

In [ ]:
vars_ = X_total.var(axis=0)
q1 = np.quantile(vars_, 0.25) 
selector = VarianceThreshold(threshold = q1)
#preprocessor = make_pipeline(selector)

#preprocessor = make_pipeline(SelectKBest(k=300))
#preprocessor = make_pipeline(SelectKBest(score_func=f_classif, k=300))
#preprocessor = make_pipeline(SelectKBest(score_func=mutual_info_classif, k))


LR_L2 = make_pipeline(preprocessor, StandardScaler(), LogisticRegression(penalty='l2', random_state = 0, solver = 'liblinear'))
LR_L1 = make_pipeline(preprocessor, StandardScaler(), LogisticRegression(penalty='l1', random_state = 0, solver = 'liblinear'))
LR_EN = make_pipeline(preprocessor, StandardScaler(), LogisticRegression(penalty='elasticnet', l1_ratio = 0.5, random_state = 0, solver = 'saga'))

for i in range(100,375):
    preprocessor = SelectKBest(score_func=mutual_info_classif, k=i)
    LR_L1 = make_pipeline(preprocessor, StandardScaler(), LogisticRegression(penalty='l1', random_state = 0, solver = 'liblinear'))
    print("------" , i , "-------")
    evaluation_2(LR_L1)

In [ ]:
vars_ = X_total.var(axis=0)
q1 = np.quantile(vars_, 0.75) 
selector = VarianceThreshold(threshold = q1)
#preprocessor = make_pipeline(selector)

#preprocessor = make_pipeline(SelectKBest(k=350))
preprocessor = SelectKBest(score_func=f_classif, k=105)


RandomForest = make_pipeline(preprocessor, RandomForestClassifier(random_state = 0))
AdaBoost = make_pipeline(preprocessor, AdaBoostClassifier(random_state = 0))
GB = make_pipeline(preprocessor, GradientBoostingClassifier(random_state = 0))

evaluation(RandomForest)
evaluation(AdaBoost)
evaluation(GB)

for i in range(100,375):
    preprocessor = SelectKBest(score_func=f_classif, k=i)
    RandomForest = make_pipeline(preprocessor, RandomForestClassifier(random_state = 0))
    print("------" , i , "-------")
    evaluation_2(RandomForest)

# Bootstrap

In [23]:
vars_ = X_total.var(axis=0)
q1 = np.quantile(vars_, 0.25) 
selector = VarianceThreshold(threshold = q1)
preprocessor = make_pipeline(selector)

#preprocessor = make_pipeline(SelectKBest(k=300))
#preprocessor = make_pipeline(SelectKBest(score_func=f_classif, k=155))
    

LR_L2 = make_pipeline(preprocessor, StandardScaler(), LogisticRegression(penalty='l2', random_state = 0, solver = 'liblinear'))
LR_L1 = make_pipeline(preprocessor, StandardScaler(), LogisticRegression(penalty='l1', random_state = 0, solver = 'liblinear'))
LR_EN = make_pipeline(preprocessor, StandardScaler(), LogisticRegression(penalty='elasticnet', l1_ratio = 0.5, random_state = 0, solver = 'saga'))
evaluation(LR_L1)
evaluation(LR_L2)
evaluation(LR_EN)

[[3 0]
 [0 3]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00         3

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6

[[3 0]
 [0 3]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00         3

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6

[[3 0]
 [0 3]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00         3

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      

C:\Users\tamer\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [24]:
import numpy as np
import pandas as pd

model = LR_L1

y_arr = np.asarray(y_total)  # <- important fix

n_boot = 1000
n_features = X_total.shape[1]
coef_mat = np.zeros((n_boot, n_features))

for b in range(n_boot):

    idx_R = np.where(y_arr == 1)[0]
    idx_S = np.where(y_arr == 0)[0]

    boot_R = np.random.choice(idx_R, size=len(idx_R), replace=True)
    boot_S = np.random.choice(idx_S, size=len(idx_S), replace=True)

    boot_idx = np.concatenate([boot_R, boot_S])

    Xb = X_total.iloc[boot_idx]
    yb = y_arr[boot_idx]  # <- now positional, no KeyError

    model.fit(Xb, yb)

    coefs = model.named_steps["logisticregression"].coef_[0]
    support = model.named_steps['pipeline'].named_steps['variancethreshold'].get_support()

    full_coef = np.zeros(n_features)
    full_coef[support] = coefs

    coef_mat[b, :] = full_coef


In [25]:
print(coef_mat.shape)
print("Any non-zero coefficients?", np.any(coef_mat != 0))
print("Mean #nonzero per bootstrap:", np.mean(np.sum(coef_mat != 0, axis=1)))
print("Std  #nonzero per bootstrap:", np.std(np.sum(coef_mat != 0, axis=1)))


(1000, 375)
Any non-zero coefficients? True
Mean #nonzero per bootstrap: 43.21
Std  #nonzero per bootstrap: 12.028378943149404


In [26]:
coef_df = pd.DataFrame(coef_mat, columns=X_total.columns)

summary = pd.DataFrame({
    "metabolite": X_total.columns,
    "mean_coef": coef_df.mean(axis=0),
    "std_coef": coef_df.std(axis=0),
    # fraction of bootstraps where coefficient != 0 (usually equals "passed variance filter")
    "nonzero_freq": (coef_df != 0).mean(axis=0),
    # fraction of bootstraps where coefficient is positive / negative (ignoring zeros)
    "pos_freq": (coef_df > 0).mean(axis=0),
    "neg_freq": (coef_df < 0).mean(axis=0),
})

# sign consistency (close to 1 means always same sign; close to 0 means flips)
summary["sign_consistency"] = (summary["pos_freq"] - summary["neg_freq"]).abs()

# optional: "signal-to-noise" style ranking
summary["abs_mean_over_sd"] = summary["mean_coef"].abs() / (summary["std_coef"] + 1e-12)

# sort: first by stability, then by magnitude/stability ratio
summary = summary.sort_values(
    ["nonzero_freq", "sign_consistency", "abs_mean_over_sd"],
    ascending=[False, False, False]
).reset_index(drop=True)

print(summary.head(20))


                                           metabolite  mean_coef  std_coef  \
0                                          Myricoside  -0.062937  0.074379   
1   (E)-3-methyl-6-(pent-3-en-1-yn-1-yl)-1,2-dithiine  -0.056694  0.050491   
2                                       Adiantifoline   0.056022  0.061830   
3                   Osthenol-7-O-beta-D-gentiobioside  -0.039617  0.043798   
4                                             Didymin  -0.044828  0.048802   
5                                        Thalicarpine   0.065013  0.082700   
6                                  2-Oxo-turkesterone   0.045468  0.047194   
7                                 trans-Aconitic acid   0.045887  0.052726   
8   Kaempferol 3-apiosyl-(1->4)-rhamnoside-7-rhamn...  -0.030212  0.054149   
9              Genistein 7,4'-bis(O-glucosylapioside)  -0.033724  0.041327   
10                                         Senkirkine   0.045196  0.073973   
11                                  Methyl hesperidin  -0.026572

In [27]:
summary.to_excel("C:/Users/tamer/Documents/PhD/ML/ML_coefs/bootstrap.xlsx", index=False)
print("Saved: bootstrap_LR_L2_coeff_stability.xlsx")


Saved: bootstrap_LR_L2_coeff_stability.xlsx
